# remembox — Quickstart

**Production-grade plug-and-play memory for AI agents.**
Zero config, single-file storage, framework-agnostic.

This notebook gets you from `pip install` to a memory-augmented prompt in **5 minutes**.

> Run cells top to bottom. Everything uses an in-memory database (`":memory:"`) so no files are created on disk.


## 0. Install & import

Install from source (the package lives in `src/`):

```bash
uv sync --extra dev
```


In [ ]:
# Make the local package importable when running this notebook from the repo root.
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

from remembox import Remembox, MemoryConfig


## 1. Create a memory

One class, one import. Backed by SQLite (WAL mode). State persists across restarts when you point it at a real file path.


In [ ]:
memory = Remembox(":memory:")  # use "my_agent.db" to persist to disk
memory

## 2. Record events (episodic memory)

`record()` stores a timestamped event with an **importance** score (0.0 trivial → 1.0 life-changing) and an optional emotion tag.


In [ ]:
memory.record("User got promoted to Director of Engineering!", importance=1.0, emotion="ecstatic")
memory.record("User ordered black coffee", importance=0.3)
memory.record("User's dog Rocky had a vet visit. All clear.", importance=0.5, emotion="relieved")

# Peek at the most recent events
for ep in memory.recent(3):
    print(f"[{ep.emotion or '-':9}] imp={ep.importance:.1f} | {ep.content}")


## 3. Learn facts (semantic memory)

`learn()` stores a `(subject, predicate, object)` triple with automatic **reinforcement** (repeat a fact → confidence rises) and **contradiction** handling (new value → old value deactivated).


In [ ]:
memory.learn("user", "name",      "Pranav",      confidence=0.95)
memory.learn("user", "prefers",   "black coffee", confidence=0.9)

# Reinforcement: same fact again → confidence climbs toward 1.0
fact, action = memory.learn("user", "prefers", "black coffee")
print(f"action={action}, new confidence={fact.confidence:.3f}")

# Contradiction: new value → old fact deactivated, new one becomes active
memory.learn("user", "lives_in", "Delhi", confidence=0.8)
memory.learn("user", "lives_in", "Mumbai")
print("lives_in facts:", [(f.object, f.is_active) for f in memory.about("user") if f.predicate == "lives_in"])


## 4. Retrieve memories (smart recall)

`recall()` scores episodes by **Recency × Relevance × Importance** and returns the top-k with a score breakdown.


In [ ]:
results = memory.recall("coffee", k=3)
for r in results:
    print(f"score={r.score:.3f}  R={r.recency:.2f} V={r.relevance:.2f} I={r.importance:.2f}")
    print(f"   → {r.episode.content}")


## 5. Build LLM context

`context()` returns a **prompt-ready string** — user profile + relevant memories — that fits inside a token budget. Paste it straight into your system prompt.


In [ ]:
context = memory.context("What does the user like?", max_tokens=2000)
print(context)


In [ ]:
# Plug into any LLM (OpenAI shown; works with Anthropic, Ollama, OpenRouter, ...)
messages = [
    {"role": "system", "content": f"You are a helpful assistant.\n\n{context}"},
    {"role": "user",   "content": "What should I get for dinner?"},
]
messages  # send to: client.chat.completions.create(model=..., messages=messages)


## 6. Maintenance

Two background jobs keep memory healthy over time:

- `consolidate()` — compress old episodes into durable facts
- `forget()` — prune/archive stale memories (importance-weighted; critical ones survive)


In [ ]:
print("consolidate:", memory.consolidate())
print("forget:      ", memory.forget())
print("stats:       ", memory.stats()["episodes"])


## ✅ You're done

You now know the full loop:

```
record → learn → recall → context → [LLM] → record → ... → consolidate / forget
```

**Next:**
- `notebooks/walkthrough.ipynb` — a detailed simple→advanced tour (procedural memory, threads, reflection, temporal facts, embeddings, multi-user, LLM integration).
- `README.md` — full API reference & design notes.
- `demos/demo_openrouter.py` — a live end-to-end demo with a real LLM.
